# MAINTAIN AI — Predictive Maintenance Bootstrap Training

This notebook is the reproducible Google Colab entry point for the first real-data training cycle. It downloads/loads public PHM data, prepares temporal sequences, trains the shared category-aware model, evaluates held-out assets, and exports the model bundle.

**Important:** this bootstrap model is not yet the MAINTAIN AI 24h/48h/7d failure model. Public datasets use different physical systems and time units. We keep their semantics intact. The production failure-risk labels will later come from real MAINTAIN AI timestamps and technician-confirmed outcomes.

In [ ]:
# 1. Clone Lab and install dependencies
!git clone -b Lab https://github.com/jadhavdurvesh/Maintain.ai.3.git /content/Maintain.ai.3
%cd /content/Maintain.ai.3
!pip install -q -r training/requirements.txt

In [ ]:
# 2. Check GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: enable a Colab GPU runtime before training.')

## 3. Dataset locations

Upload/download the real datasets into `/content/training-data` using their official distribution/access process. Do not commit raw datasets to GitHub. The first supported bootstrap sources are IMS bearings, Paderborn, CWRU and NASA C-MAPSS.

In [ ]:
from pathlib import Path
DATA=Path('/content/training-data')
PREP=Path('/content/prepared'); SPLIT=Path('/content/splits'); SEQ=Path('/content/sequences'); ART=Path('/content/artifacts')
for p in (DATA,PREP,SPLIT,SEQ,ART): p.mkdir(parents=True,exist_ok=True)
print(DATA)

## 4. Convert a dataset

Run only the adapters for datasets you have actually downloaded. C-MAPSS is the cleanest first run because it provides explicit run-to-failure trajectories.

In [ ]:
# Example: C-MAPSS FD001
!PYTHONPATH=training/src python -m adapters.build_common \
  --dataset cmapss \
  --path /content/training-data/cmapss/train_FD001.txt \
  --out /content/prepared/cmapss_FD001.parquet

In [ ]:
# Optional: inspect the normalized data before training
import pandas as pd
df=pd.read_parquet('/content/prepared/cmapss_FD001.parquet')
print('rows:',len(df))
print('assets:',df.asset_id.nunique())
print('columns:',len(df.columns))
display(df.head())

In [ ]:
# 5. Split by physical asset/run — prevents leakage
!PYTHONPATH=training/src python training/src/split_assets.py \
  --input /content/prepared/cmapss_FD001.parquet \
  --out-dir /content/splits/cmapss

In [ ]:
# 6. Build temporal sequences
!PYTHONPATH=training/src python training/src/build_sequences.py \
  --input /content/splits/cmapss/train.parquet \
  --out /content/sequences/cmapss_train.pt \
  --sequence-length 24

!PYTHONPATH=training/src python training/src/build_sequences.py \
  --input /content/splits/cmapss/test.parquet \
  --out /content/sequences/cmapss_test.pt \
  --sequence-length 24

In [ ]:
# 7. Train the shared temporal model
!PYTHONPATH=training/src python training/src/train.py \
  --data /content/sequences/cmapss_train.pt \
  --epochs 50 \
  --batch-size 128 \
  --lr 0.0003 \
  --out /content/artifacts/shared_temporal_v1.pt

In [ ]:
# 8. Evaluate on unseen assets
!PYTHONPATH=training/src python training/src/evaluate.py \
  --data /content/sequences/cmapss_test.pt \
  --model /content/artifacts/shared_temporal_v1.pt \
  --out /content/artifacts/cmapss_test_metrics.json

import json
print(json.dumps(json.load(open('/content/artifacts/cmapss_test_metrics.json')),indent=2))

## 9. Export the model bundle

The checkpoint should travel with its configuration and evaluation metrics. Do not deploy a checkpoint without its preprocessing/model metadata.

In [ ]:
import json, shutil
bundle=ART/'maintain_ai_shared_temporal_v1'
bundle.mkdir(exist_ok=True)
shutil.copy2(ART/'shared_temporal_v1.pt',bundle/'model.pt')
shutil.copy2(ART/'cmapss_test_metrics.json',bundle/'metrics.json')
shutil.copy2('training/config.yaml',bundle/'training_config.yaml')
metadata={
  'model_version':'shared-temporal-v1',
  'bootstrap_source':'nasa_cmapss',
  'sequence_length':24,
  'category_mapping':{'induction_motor':0,'pump':1,'compressor':2,'conveyor':3,'other':4},
  'note':'C-MAPSS RUL is source-native cycles; not MAINTAIN AI 24h/48h/7d failure risk.'
}
(bundle/'model_metadata.json').write_text(json.dumps(metadata,indent=2))
!cd /content && zip -qr maintain_ai_shared_temporal_v1.zip artifacts/maintain_ai_shared_temporal_v1
print('Bundle:',bundle)


## 10. Before deployment

This checkpoint is a **bootstrap temporal/RUL model**. It must not be presented to users as a validated 24h/48h/7d machine-failure predictor. Before production deployment we need verified category-specific failure datasets, real MAINTAIN AI telemetry, technician-confirmed outcomes, timestamp-aware labels, calibration, and asset-level holdout evaluation.

In [ ]:
# Optional: download the exported bundle from Colab
from google.colab import files
files.download('/content/maintain_ai_shared_temporal_v1.zip')